# HADES sweeps: Morris screening

Many HADES runs for Paper 2. This notebook runs the **screening stage**: which uncertain parameters move the impacts of Pathway A, Pathway B and their difference, so that the main sweep samples only those (decision log, section F).

**How it runs** (`ADM1/sweep_engine.py`): workers build each run in memory and read ecoinvent only, so many can run in parallel without Brightway database writes. Each point (feedstock, T, RT, parameter sample, pathway) runs ADM1 once and is then evaluated for every LCA context and heat configuration. Impacts come from unit scores computed once in step 3 and checked against a full Brightway LCA in step 4. Every finished point is saved to its own file, so an interrupted run resumes where it stopped.

**Design.** Morris elementary effects (Morris 1991, Technometrics 33:161), r trajectories on a 4-level grid, mu* as importance (Campolongo, Cariboni & Saltelli 2007, Environ. Model. Softw. 22:1509). Both pathways use the same samples (paired).

Factors: 15 uncertain parameters and, with `INCLUDE_DESIGN`, 4 design variables (HRT, HTC feed moisture, process-water COD, digester feed TS). T, RT, province and heat supply stay on a grid. Dilution follows operator practice: a feed drier than a target gets water, a wetter one is never dried (only the Pathway B digestate is mechanically dewatered).

**Limit.** The HTC surrogate's prediction error is not sampled (decision log F2).

Steps: 1 settings · 2 design · 3 unit scores · 4 check against Brightway · 5 timing · 6 run · 7 results.

## 1. Settings

In [ ]:
PROJECT = "my_hades_project"                 # Brightway project
EI_NAME = "ecoinvent-3.12-cutoff"
WORKERS = 12                                 # parallel workers (16 cores, 15 GB RAM: 12 leaves headroom)
SWEEP_NAME = "morris_v1"

IW_MID = "IMPACT World+ Midpoint 2.2.1_regionalized for ecoinvent v3.12"
IW_DMG = "IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12"
METHODS = {"GWP100 (kg CO2 eq)":            (IW_MID, "Midpoint", "Climate change, short term"),
           "Human health (DALY)":           (IW_DMG, "Human health", "Total human health"),
           "Ecosystem quality (PDF.m2.yr)": (IW_DMG, "Ecosystem quality", "Total ecosystem quality")}

# screening scenarios (the main sweep uses all provinces)
CONTEXTS = ["QC", "ON", "AB"]                # lca_contexts presets
HEAT_CONFIGS = ["paper1", "all_ng", "own_biogas"]
T_LEVELS = [180, 260]                        # C
RT_LEVELS = [60]                             # min

# Morris design
INCLUDE_DESIGN = True                        # also screen HRT 20-40 d, HTC feed moisture 0.75-0.90,
                                             # process-water COD 5-40 kg/m3 (A), digester feed TS 5-12 % (B)
INCLUDE_FEED_MOISTURE = True                 # sample feed moisture within each feedstock's WC_range
MORRIS_R = 20                                # trajectories
MORRIS_LEVELS = 4
SEED = 20261123

# feedstocks: composition % dry (O by difference), WC % wet (central), WC_range (sampled when
# INCLUDE_FEED_MOISTURE). A feedstock with WC None is skipped.
FEEDS = {
    "food_waste": dict(feedstock_type="food_waste",
                       feed=dict(WC=85.0, C_feed=38.25, H_feed=5.47, N_feed=3.66, S_feed=0.50, Ash_feed=9.10,
                                 VM_feed=82.6, FC_feed=8.3),
                       WC_range=(74.0, 90.0),
                       source="Paper 1 base case; WC range: Zhang et al. 2007, Bioresour. Technol. 98, 929"),
    "ofmsw": dict(feedstock_type="ofmsw",
                  feed=dict(WC=70.0, C_feed=45.00, H_feed=6.00, N_feed=2.50, S_feed=0.20, Ash_feed=5.30,
                            VM_feed=78.0, FC_feed=16.7),
                  WC_range=(50.0, 92.0),
                  source=("Paper 1 case study; WC range: as-received OFMSW characterizations, 50 to 92 % "
                          "depending on collection system (67-70 % in a one-year German study)")),
    "sewage_sludge": dict(feedstock_type="sewage_sludge",
                          WC_range=(71.0, 85.0),
                          feed=dict(WC=78.0, C_feed=31.25, H_feed=4.52, N_feed=4.14, S_feed=1.75, Ash_feed=40.26,
                                    VM_feed=52.32, FC_feed=7.42),
                          source=("mean of 13 compositions (9 studies), HTC training dataset, rows with ash < 1 % "
                                  "excluded; VM/FC from the mean FC/(VM+FC) of 12 rows; WC: mechanically dewatered sludge "
                                  "71-85 % (Energies, MDPI, wastewater sludge characterization), central = midpoint")),
    "dairy_manure": dict(feedstock_type="dairy_manure",
                         WC_range=(83.0, 88.0),
                         feed=dict(WC=85.0, C_feed=37.80, H_feed=5.03, N_feed=2.37, S_feed=0.47, Ash_feed=25.37,
                                   VM_feed=60.77, FC_feed=13.87),
                         source=("mean of 3 raw manure compositions (2 studies: Belete et al. 2021, doi:10.1016/"
                                 "j.biortech.2021.125164, dairy; doi:10.1016/j.renene.2020.07.003), HTC dataset, digestate "
                                 "rows excluded; VM/FC from the single row reporting them; WC: fresh dairy manure 83-84 % "
                                 "(16-17 % TS, lab studies), 85.7 % as excreted (Cornell), about 88 % (MWPS-18); "
                                 "central = mean")),
}

## 2. Design

In [ ]:
import os, sys, json, importlib.util
import numpy as np
import pandas as pd

ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "ADM1")) else os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "ADM1"))
import sweep_engine as se                    # workers import it by name, so it must be importable
sb = se._load("system_builder", "system_builder.py")
OUT = os.path.join(ROOT, "results", "sweeps", SWEEP_NAME)
SCORES = os.path.join(OUT, "unit_scores.json")
POINTS_DIR = os.path.join(OUT, "points")
os.makedirs(POINTS_DIR, exist_ok=True)

feeds = {}
for k, f in FEEDS.items():
    if f["feed"]["WC"] is None:
        print(f"skipped {k}: no moisture (WC) given")
        continue
    d = f["feed"]
    if d["C_feed"] + d["H_feed"] + d["N_feed"] + d["S_feed"] + d["Ash_feed"] >= 100:
        raise ValueError(f"{k}: C + H + N + S + ash >= 100 %")
    if abs(d["VM_feed"] + d["FC_feed"] + d["Ash_feed"] - 100) > 1.5:
        print(f"WARNING {k}: VM + FC + ash = {d['VM_feed'] + d['FC_feed'] + d['Ash_feed']:.1f} %")
    feeds[k] = f

space = se.parameter_space(sb, sb.hce, sb._adm1_core, include_design=INCLUDE_DESIGN,
                           include_feed_moisture=INCLUDE_FEED_MOISTURE)
names = [s["name"] for s in space]
X, steps = se.morris_design(len(space), MORRIS_R, MORRIS_LEVELS, seed=SEED)
points = se.make_points(X, feeds, T_LEVELS, RT_LEVELS, ["A", "B"], space, sb._adm1_core, prefix="m")
design = dict(names=names, X=X.tolist(), steps=steps, seed=SEED, levels=MORRIS_LEVELS, feeds=FEEDS,
              contexts=CONTEXTS, heat_configs=HEAT_CONFIGS, T=T_LEVELS, RT=RT_LEVELS)
design = json.loads(json.dumps(design, default=str))
dpath = os.path.join(OUT, "design.json")
if os.path.exists(dpath) and json.load(open(dpath)) != design and os.listdir(POINTS_DIR):
    raise RuntimeError(f"{OUT} holds results of a different design. Change SWEEP_NAME in cell 1 "
                       "(or delete that folder) so old points are not reused.")
json.dump(design, open(dpath, "w"), indent=1)
print(f"{len(space)} factors: {names}")
print(f"{len(X)} samples x {len(feeds)} feedstocks x {len(T_LEVELS) * len(RT_LEVELS)} conditions x 2 pathways "
      f"= {len(points)} points, each for {len(CONTEXTS)} contexts x {len(HEAT_CONFIGS)} heat configurations")

## 3. Unit scores (once per design, serial)
Probe runs at the centre of every range collect all background datasets; their unit scores and the characterization factors of the foreground emissions are stored in `unit_scores.json`. Re-run this cell if step 7 reports missing keys.

In [ ]:
import bw2data as bd
probes = se.make_points(np.full((1, len(space)), 0.5), feeds, [min(T_LEVELS), max(T_LEVELS)], RT_LEVELS,
                        ["A", "B"], space, sb._adm1_core, prefix="probe")
scores = se.precompute_scores(probes, HEAT_CONFIGS, CONTEXTS, list(METHODS.values()), PROJECT, EI_NAME, SCORES)

## 4. Check the engine against a full Brightway LCA
Two probe points, rebuilt in a real foreground database and computed with bw2calc. Stops if any impact differs by more than 0.01 %.

In [ ]:
for pt, ctx_, heat in [(probes[0], "ON", "own_biogas"), (probes[1], "QC", "paper1")]:
    chk = se.validate_point(pt, ctx_, heat, METHODS, PROJECT, EI_NAME, SCORES)
    print(pt["point_id"], ctx_, heat); display(chk)

## 5. Timing
Runs one point per worker and estimates the full run.

In [ ]:
import time
LABELS = se.method_labels_for(METHODS)
t0 = time.time()
se.run_sweep(points[:WORKERS], HEAT_CONFIGS, CONTEXTS, LABELS, PROJECT, EI_NAME, SCORES, POINTS_DIR, workers=WORKERS)
dt = time.time() - t0
print(f"{WORKERS} points in {dt / 60:.1f} min -> full design about {dt * len(points) / WORKERS / 3600:.1f} h "
      f"(includes worker start-up, so this overestimates)")

## 6. Run
Resumes automatically: points already in `points/` are skipped.

In [ ]:
se.run_sweep(points, HEAT_CONFIGS, CONTEXTS, LABELS, PROJECT, EI_NAME, SCORES, POINTS_DIR, workers=WORKERS)

## 7. Results: Morris importance
mu* per factor for Pathway A, Pathway B and the paired difference A − B (negative = A better), per scenario, then summarized over scenarios as the median of mu* scaled by the largest mu* of that scenario.

In [ ]:
df = se.collect(POINTS_DIR)
if "error" in df:
    err = df[df["error"].notna()]
    print(f"{len(err)} failed evaluations"); display(err[["point_id", "context", "heat_config", "error"]].head(20))
    df = df[df["error"].isna()]
miss = df["missing_keys"].fillna("")
print(f"{(miss != '').sum()} evaluations with missing unit scores (re-run step 3)")

KEYS = ["feedstock", "T", "RT", "context", "heat_config", "sample_id"]
summary = {}
for lab in METHODS:
    w = df.pivot_table(index=KEYS, columns="pathway", values=lab).reset_index()
    w["A - B"] = w["A"] - w["B"]
    rows = []
    for scen, g in w.groupby(KEYS[:-1]):
        for out in ("A", "B", "A - B"):
            y = g.set_index("sample_id")[out].reindex(range(len(X))).to_numpy()
            mi = se.morris_indices(y, steps, names)
            mi["scaled"] = mi["mu_star"] / mi["mu_star"].max()
            for _, r in mi.iterrows():
                rows.append(dict(zip(KEYS[:-1], scen), output=out, **r))
    summary[lab] = pd.DataFrame(rows)
    s = summary[lab].groupby(["output", "factor"])["scaled"].median().unstack("output")
    print(f"\n{lab}: median scaled mu* over {w.groupby(KEYS[:-1]).ngroups} scenarios")
    display(s.sort_values("A - B", ascending=False).style.format("{:.2f}"))
pd.concat([d.assign(method=k) for k, d in summary.items()]).to_csv(os.path.join(OUT, "morris_indices.csv"), index=False)